In [2]:
import json
import os
import re
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

In [3]:


# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42
USE_SUPERVISED = True

# MLP — основная ML-модель прогноза.
# geo_score используется только как один из входных признаков,
# а итоговую карту строит model.predict(...).
MLP_HIDDEN = (8, 4)
MLP_ALPHA = 12.0
MLP_MAX_ITER = 6000
MLP_LEARNING_RATE_INIT = 0.003

# Радиус влияния известных проявлений/геохимических точек для обучающей цели.
# Чем больше значение, тем более гладкая целевая поверхность.
EVIDENCE_RADIUS_CELLS = 6.0

# Сглаживание именно ML-результата, чтобы убрать пиксельный шум.
ML_SMOOTH_PASSES = 3

Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False


In [4]:

# =========================
# PATHS
# =========================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "mlp_main_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_mlp_main.gpkg"
OUT_PNG = OUT_DIR / "forecast_mlp_main.png"
OUT_COMPARE = OUT_DIR / "compare_mlp_main.png"
OUT_PROX = OUT_DIR / "prox_magm_mlp_main.png"
OUT_CSV = OUT_DIR / "grid_mlp_main.csv"
OUT_JSON = OUT_DIR / "metrics_mlp_main.json"

In [5]:

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    """
    Выделение наиболее перспективных зон для отображения.
    Важно: это НЕ отдельный фактор итогового прогноза.
    Итоговый прогноз уже рассчитан как MLP-предсказание в поле prospectivity.
    Здесь только выбираются верхние локальные области для жёлтой подсветки.
    """
    q_core = float(grid["prospectivity"].quantile(0.955))
    q_support = float(grid["prospectivity"].quantile(0.90))

    support = grid["prospectivity"] >= q_support
    local_peak = local_max_mask(grid, "prospectivity", shape)

    # Небольшая геологическая проверка, чтобы жёлтые зоны не появлялись
    # совсем без поддержки тектоники/магматизма/структурного фактора.
    support_geo = (
        (grid["tect_magm_intersection"] >= grid["tect_magm_intersection"].quantile(0.72)) |
        (grid["tect_struct_intersection"] >= grid["tect_struct_intersection"].quantile(0.72)) |
        (grid["coincidence_score"] >= grid["coincidence_score"].quantile(0.72))
    )

    grid["gold_zone_raw"] = (
        (grid["prospectivity"] >= q_core) |
        (support & local_peak & support_geo)
    ).astype(int)

    grid["gold_zone"] = keep_large_components(grid, "gold_zone_raw", shape, min_cells=4).astype(int)
    return grid


def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(8, 8))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("Магматогенный фактор: proximity")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


In [6]:
# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

In [7]:
# =========================
# GRID + FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1)
    * np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1)
    * np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)


In [8]:
# =========================
# GEO SCORE
# =========================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

In [9]:
# =========================
# MLP MODEL — ML ОСНОВА ПРОГНОЗА
# =========================

# 1) Целевая переменная.
# Если есть точки прямых поисковых признаков, строим непрерывную обучающую цель:
# чем ближе ячейка к известной точке/аномалии, тем выше target.
# Это лучше, чем грубые классы 0/1: отсутствие точки не означает отсутствие руды.
grid["target"] = 0.0
grid["evidence_score"] = 0.0
positive_cells = []
use_supervised = False
mlp_test_proxy = None

if USE_SUPERVISED and points is not None and len(points) > 0:
    point_union = unary_union(points.geometry)
    dist_to_evidence = np.array([geom.centroid.distance(point_union) for geom in grid.geometry], dtype=float)
    radius = CELL_SIZE * EVIDENCE_RADIUS_CELLS

    # Плавная близость к известным проявлениям.
    evidence_score = np.exp(-dist_to_evidence / radius)
    evidence_score = robust_normalize_01(evidence_score, 0.02, 0.995)
    grid["evidence_score"] = evidence_score

    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()

    # Ячейки с самими точками должны иметь максимум.
    if positive_cells:
        grid.loc[grid["cell_id"].isin(positive_cells), "evidence_score"] = 1.0

    # target — это данные контроля, а не ручная формула прогноза.
    # Небольшая примесь geo_score_sm стабилизирует обучение там, где точек мало.
    grid["target"] = robust_normalize_01(
        0.85 * grid["evidence_score"] + 0.15 * grid["geo_score_sm"],
        0.02, 0.995
    )
    use_supervised = len(positive_cells) >= 3
else:
    # Запасной режим: если нет контрольных точек, MLP учится аппроксимировать
    # геологическую факторную поверхность. Это хуже, но код останется рабочим.
    grid["target"] = grid["geo_score_sm"].to_numpy()
    use_supervised = False


# 2) Входные признаки модели.
# geo_score_sm — только один входной признак, не итоговая формула.
# Итог строится через MLPRegressor.predict().
feature_cols = [
    "prox_facies",
    "prox_paleo",
    "prox_struct",
    "prox_magm",
    "prox_tect1",
    "prox_tect2",
    "tect_combo",
    "tect_intersection",
    "tect_magm_intersection",
    "tect_struct_intersection",
    "paleo_struct_intersection",
    "coincidence_score",
    "tect_only_penalty",
    "geo_score_sm",
]

X = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
y = grid["target"].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()


# 3) Обучение MLP.
# StandardScaler обязателен для нейросети.
mlp = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=MLP_HIDDEN,
        activation="relu",
        solver="adam",
        alpha=MLP_ALPHA,
        learning_rate_init=MLP_LEARNING_RATE_INIT,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=80,
        random_state=RANDOM_STATE,
    )
)

# Простая проверка: обучаемся на части территории, смотрим,
# насколько выше предсказания около известных точек, чем вдали.
if len(grid) > 30:
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE
        )
        mlp_eval = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=MLP_HIDDEN,
                activation="relu",
                solver="adam",
                alpha=MLP_ALPHA,
                learning_rate_init=MLP_LEARNING_RATE_INIT,
                max_iter=MLP_MAX_ITER,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=80,
                random_state=RANDOM_STATE,
            )
        )
        mlp_eval.fit(X_train, y_train)
        pred_test = np.clip(mlp_eval.predict(X_test), 0, 1)
        high = y_test >= np.quantile(y_test, 0.80)
        low = y_test <= np.quantile(y_test, 0.20)
        if high.any() and low.any():
            mlp_test_proxy = float(np.mean(pred_test[high]) - np.mean(pred_test[low]))
    except Exception as e:
        mlp_test_proxy = None

mlp.fit(X, y)
grid["mlp_raw"] = np.clip(mlp.predict(X), 0, 1)
grid["mlp_score"] = robust_normalize_01(grid["mlp_raw"], 0.02, 0.98)
grid["mlp_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "mlp_score", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02, 0.98
)

# Важность признаков для MLP напрямую не считается.
# Поэтому используем простую permutation-оценку: насколько меняется прогноз,
# если перемешать один признак.
rng = np.random.default_rng(RANDOM_STATE)
base_pred = grid["mlp_raw"].to_numpy()
feature_importance = {}
for j, col in enumerate(feature_cols):
    X_perm = X.copy()
    X_perm[:, j] = rng.permutation(X_perm[:, j])
    pred_perm = np.clip(mlp.predict(X_perm), 0, 1)
    feature_importance[col] = float(np.mean(np.abs(base_pred - pred_perm)))

feature_importance = dict(sorted(feature_importance.items(), key=lambda kv: kv[1], reverse=True))


In [10]:
# =========================
# FINAL SURFACE
# =========================

# Итоговая поверхность строится ML-моделью.
# Здесь нет суммы geo_score + ml_score: geo_score уже был только входным признаком MLP.
grid["prospectivity_raw"] = grid["mlp_score_sm"]

# Небольшой штраф у границ нужен только против краевых артефактов,
# он не является отдельным прогнозным фактором.
mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.025 * grid["edge_false_penalty"]
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=1)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# Для совместимости с методичкой:
# в поле prognoz меньшее значение означает более перспективную ячейку.
grid["prognoz"] = 1.0 - grid["prospectivity"]

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)


In [11]:
# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
        "positive_cells": int(len(positive_cells)),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "mlp_test_proxy": None if mlp_test_proxy is None else float(mlp_test_proxy),
    "point_count": int(len(points)) if points is not None else 0,
    "mlp_permutation_importance": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print(grid[["prospectivity", "prognoz", "mlp_score_sm", "gold_zone"]].describe())


Готово.
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\forecast_rf_clean.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\compare_rf_clean.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\forecast_rf_clean.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\grid_rf_clean.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\metrics_rf_clean.json
       prospectivity       prognoz   rf_score_sm     gold_zone
count   15684.000000  15684.000000  15684.000000  15684.000000
mean        0.357636      0.642364      0.190389      0.004017
std         0.231341      0.231341      0.237673      0.063253
min         0.000000      0.000000      0.000000      0.000000
25%         0.183806      0.507615      0.029985      0.000000
50%         0.328987      0.671013      0.096525      0.000000
75%         0.492385      0.816194      0.242892      0.000000
max         1.000000      1.000000  